同步阻塞代码导致async失败

In [2]:
import asyncio
import time

async def bad_task():
    print(f"[bad]  开始 sleep @ {time.strftime('%X')}")
    time.sleep(3)                       # 💀 同步阻塞 3 秒
    print(f"[bad]  结束 sleep @ {time.strftime('%X')}")

async def good_task():
    for i in range(6):
        print(f"[good] tick {i} @ {time.strftime('%X')}")
        await asyncio.sleep(0.5)        # ✅ 异步 sleep

async def main():
    await asyncio.gather(bad_task(), good_task())

await main() ## notebook 里直接 await，不要用 asyncio.run()


[bad]  开始 sleep @ 15:16:49
[bad]  结束 sleep @ 15:16:52
[good] tick 0 @ 15:16:52
[good] tick 1 @ 15:16:53
[good] tick 2 @ 15:16:53
[good] tick 3 @ 15:16:54
[good] tick 4 @ 15:16:54
[good] tick 5 @ 15:16:55


并发HTTP请求示例

In [ ]:
import asyncio
import aiohttp

async def fetch(session, url):
    async with session.get(url) as resp:
        return await resp.text()

async def main():
    urls = [f"https://example.com/{i}" for i in range(100)]
    async with aiohttp.ClientSession() as session:
        results = await asyncio.gather(
            *[fetch(session, url) for url in urls]
        )
    print(len(results), len(results[0]))
    
await main()


100 559


模仿 asyncio.sleep 的最简实现

In [6]:
import asyncio
import time

class MySleep:
    """等待 N 秒的自定义 awaitable"""
    def __init__(self, seconds):
        self.seconds = seconds

    def __await__(self):
        # 创建一个 Future，让事件循环在 N 秒后设置结果
        loop = asyncio.get_event_loop()
        future = loop.create_future()
        loop.call_later(self.seconds, future.set_result, None)
        
        # 关键：yield future 让出控制权
        # 事件循环知道"我在等 future"，N 秒后 future 被设置，恢复我
        yield future
        
        return None

async def main():
    print(f"开始 @ {time.strftime('%X')}")
    await MySleep(2)
    print(f"结束 @ {time.strftime('%X')}")

await main()


开始 @ 16:10:51


RuntimeError: yield was used instead of yield from in task <Task pending name='Task-884' coro=<InteractiveShell.run_cell_async() running at C:\Users\PenghaoChang\AppData\Roaming\Python\Python312\site-packages\IPython\core\interactiveshell.py:3365> cb=[IPythonKernel._cancel_on_sigint.<locals>.cancel_unless_done(<Future pendi...ernel.py:329]>)() at C:\Users\PenghaoChang\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py:329, Task.task_wakeup()]> with <Future pending>

<frozen genericpath>:89: RuntimeWarning: coroutine 'main' was never awaited
